In [5]:
import numpy as np
import time

def solve_mfg_stable(Nx=100, Nt=100, T=1.0, c=10.0, kappa=5.0,
                     tol=1e-10, max_iter=100, alpha=0.9):
    dx = 1.0 / (Nx - 1)
    dt = T / (Nt - 1)
    x = np.linspace(0, 1, Nx)

    rho = np.ones((Nt, Nx))
    rho[0, :] = 1.0

    V = np.zeros((Nt, Nx))
    V[-1, :] = c * (1 - x)

    start_time = time.time()
    iterations_done = 0

    for iteration in range(max_iter):
        rho_old = rho.copy()

        V_new = V.copy()
        for n in range(Nt - 2, -1, -1):
            for i in range(1, Nx - 1):
                Vx = (V_new[n+1, i+1] - V_new[n+1, i-1]) / (2 * dx)
                v_star = np.clip(-Vx, 0.0, 5.0)
                F_rho = kappa * rho[n, i]
                V_new[n, i] = V_new[n+1, i] - dt * (
                    -0.5 * v_star**2 - F_rho + Vx * v_star
                )
            V_new[n, 0] = V_new[n, 1]
            V_new[n, -1] = V_new[n, -2]

        v_opt = np.zeros((Nt, Nx))
        for n in range(Nt):
            for i in range(1, Nx - 1):
                Vx = (V_new[n, i+1] - V_new[n, i-1]) / (2 * dx)
                v_opt[n, i] = np.clip(-Vx, 0.0, 5.0)
            v_opt[n, 0] = v_opt[n, 1]
            v_opt[n, -1] = v_opt[n, -2]

        v_max = np.max(v_opt)
        cfl = v_max * dt / dx
        if cfl > 1.0:
            local_dt = 0.9 * dx / v_max
        else:
            local_dt = dt

        rho_new = np.zeros_like(rho)
        rho_new[0, :] = 1.0

        for n in range(Nt - 1):
            for i in range(1, Nx - 1):
                if v_opt[n, i] >= 0:
                    flux_right = v_opt[n, i] * rho_new[n, i]
                    flux_left = v_opt[n, i-1] * rho_new[n, i-1] if i > 1 else 0
                    rho_new[n+1, i] = rho_new[n, i] - local_dt/dx * (flux_right - flux_left)
                else:
                    flux_left = -v_opt[n, i] * rho_new[n, i]
                    flux_right = -v_opt[n, i+1] * rho_new[n, i+1] if i < Nx-2 else 0
                    rho_new[n+1, i] = rho_new[n, i] - local_dt/dx * (flux_left - flux_right)

            rho_new[n+1, 0] = rho_new[n+1, 1]
            rho_new[n+1, -1] = rho_new[n+1, -2]
            mass = np.sum(rho_new[n+1, :]) * dx
            if mass > 0:
                rho_new[n+1, :] *= 1.0 / mass

        rho = alpha * rho_new + (1 - alpha) * rho_old

        diff = np.max(np.abs(rho - rho_old))
        iterations_done = iteration + 1
        if diff < tol:
            break

    elapsed = time.time() - start_time
    return elapsed, iterations_done, rho, v_opt


def solve_nmfg_stable(Nx=100, Nt=100, T=1.0, c=10.0, kappa_A=8.0, kappa_B=2.0,
                      tol=1e-10, max_iter=100, alpha=0.9):
    dx = 1.0 / (Nx - 1)
    dt = T / (Nt - 1)
    x = np.linspace(0, 1, Nx)

    rho_A = np.ones((Nt, Nx)) * 0.5
    rho_B = np.ones((Nt, Nx)) * 0.5
    rho_A[0, :] = 0.5
    rho_B[0, :] = 0.5

    V_A = np.zeros((Nt, Nx))
    V_B = np.zeros((Nt, Nx))
    V_A[-1, :] = c * (1 - x)
    V_B[-1, :] = c * (1 - x)

    start_time = time.time()
    iterations_done = 0

    for iteration in range(max_iter):
        rho_A_old = rho_A.copy()
        rho_B_old = rho_B.copy()
        rho_total = rho_A + rho_B

        for arr_V, kappa in [(V_A, kappa_A), (V_B, kappa_B)]:
            V_new = arr_V.copy()
            for n in range(Nt - 2, -1, -1):
                for i in range(1, Nx - 1):
                    Vx = (V_new[n+1, i+1] - V_new[n+1, i-1]) / (2 * dx)
                    v_star = np.clip(-Vx, 0.0, 5.0)
                    F_rho = kappa * rho_total[n, i]
                    V_new[n, i] = V_new[n+1, i] - dt * (
                        -0.5 * v_star**2 - F_rho + Vx * v_star
                    )
                V_new[n, 0] = V_new[n, 1]
                V_new[n, -1] = V_new[n, -2]
            arr_V[:, :] = V_new

        def compute_velocity(V_arr):
            v = np.zeros((Nt, Nx))
            for n in range(Nt):
                for i in range(1, Nx - 1):
                    Vx = (V_arr[n, i+1] - V_arr[n, i-1]) / (2 * dx)
                    v[n, i] = np.clip(-Vx, 0.0, 5.0)
                v[n, 0] = v[n, 1]
                v[n, -1] = v[n, -2]
            return v

        v_A = compute_velocity(V_A)
        v_B = compute_velocity(V_B)

        v_max = max(np.max(v_A), np.max(v_B))
        cfl = v_max * dt / dx
        local_dt = dt if cfl <= 1.0 else 0.9 * dx / v_max

        def solve_fp(v_arr, rho_init=0.5):
            rho_new = np.zeros((Nt, Nx))
            rho_new[0, :] = rho_init

            for n in range(Nt - 1):
                for i in range(1, Nx - 1):
                    if v_arr[n, i] >= 0:
                        flux_right = v_arr[n, i] * rho_new[n, i]
                        flux_left = v_arr[n, i-1] * rho_new[n, i-1] if i > 1 else 0
                        rho_new[n+1, i] = rho_new[n, i] - local_dt/dx * (flux_right - flux_left)
                    else:
                        flux_left = -v_arr[n, i] * rho_new[n, i]
                        flux_right = -v_arr[n, i+1] * rho_new[n, i+1] if i < Nx-2 else 0
                        rho_new[n+1, i] = rho_new[n, i] - local_dt/dx * (flux_left - flux_right)

                rho_new[n+1, 0] = rho_new[n+1, 1]
                rho_new[n+1, -1] = rho_new[n+1, -2]
                mass = np.sum(rho_new[n+1, :]) * dx
                if mass > 0:
                    rho_new[n+1, :] *= rho_init / mass
            return rho_new

        rho_A_new = solve_fp(v_A, 0.5)
        rho_B_new = solve_fp(v_B, 0.5)

        rho_A = alpha * rho_A_new + (1 - alpha) * rho_A_old
        rho_B = alpha * rho_B_new + (1 - alpha) * rho_B_old

        diff = max(np.max(np.abs(rho_A - rho_A_old)),
                   np.max(np.abs(rho_B - rho_B_old)))
        iterations_done = iteration + 1
        if diff < tol:
            break

    elapsed = time.time() - start_time
    return elapsed, iterations_done, rho_A, rho_B, v_A, v_B


if __name__ == "__main__":
    Nx, Nt = 100, 100
    print(f"N: {Nx} x {Nt}")
    print("=" * 50)

    n_runs = 3
    times_mfg = []
    times_nmfg = []

    for run in range(n_runs):
        print(f"\niter {run + 1}:")

        t_mfg, iter_mfg, _, _ = solve_mfg_stable(Nx=Nx, Nt=Nt, tol=1e-10, max_iter=100)
        times_mfg.append(t_mfg)
        print(f"  MFG:  {t_mfg:.2f} s, iterations: {iter_mfg}")

        t_nmfg, iter_nmfg, _, _, _, _ = solve_nmfg_stable(Nx=Nx, Nt=Nt, tol=1e-10, max_iter=100)
        times_nmfg.append(t_nmfg)
        print(f"  NMFG: {t_nmfg:.2f} s, iterations: {iter_nmfg}")

    avg_mfg = np.mean(times_mfg)
    avg_nmfg = np.mean(times_nmfg)

    print("\n" + "=" * 50)
    print("averaged results:")
    print(f"MFG:  {avg_mfg:.2f} ± {np.std(times_mfg):.2f} с")
    print(f"NMFG: {avg_nmfg:.2f} ± {np.std(times_nmfg):.2f} с")
    print(f"NMFG / MFG: {avg_nmfg / avg_mfg:.2f}")

    print(f"\ntime per iteration:")
    print(f"MFG:  {avg_mfg/iter_mfg:.4f} s/it")
    print(f"NMFG: {avg_nmfg/iter_nmfg:.4f} s/it")

N: 100 x 100

iter 1:
  MFG:  16.42 s, iterations: 100
  NMFG: 34.40 s, iterations: 100

iter 2:
  MFG:  16.57 s, iterations: 100
  NMFG: 34.68 s, iterations: 100

iter 3:
  MFG:  16.53 s, iterations: 100
  NMFG: 32.87 s, iterations: 100

averaged results:
MFG:  16.51 ± 0.06 с
NMFG: 33.98 ± 0.80 с
NMFG / MFG: 2.06

time per iteration:
MFG:  0.1651 s/it
NMFG: 0.3398 s/it
